# PDF File Metadata for Delayed Charged Lepton Source Document Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and explore a dataset defined by a Croissant schema using the `mlcroissant` library.

The workflow follows a simple pattern:
- load the Croissant metadata from its URL,
- inspect the dataset object and available record sets,
- load records using record set `@id` values,
- convert records into pandas DataFrames,
- perform lightweight exploratory analysis and visualizations.

### Dataset Source
Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.84jv-4bqg/fair2.json`

### Important note about this dataset
The provided dataset metadata indicates that the dataset currently has an empty `recordSet` list. Because all data access in Croissant should be done through entity `@id` values, this notebook first verifies whether any record sets are available.

If no record sets are present, the notebook still provides a complete, reusable exploration workflow and gracefully falls back to metadata-level inspection.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset metadata from the Croissant URL and inspect the high-level dataset description.

In this section, we deliberately keep `dataset.metadata` as a single object and access its attributes safely, without treating it like a dictionary.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

sns.set_theme(style='whitegrid')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.84jv-4bqg/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print a concise summary using attribute access
dataset_name = getattr(metadata, 'name', 'Unknown dataset')
dataset_description = getattr(metadata, 'description', 'No description available.')
dataset_id = getattr(metadata, 'id', None) or getattr(metadata, '@id', None)

print('Dataset name:', dataset_name)
print('Description:', dataset_description)
print('Dataset ID:', dataset_id)
print('Croissant URL:', croissant_url)

In [ ]:
# Optional: inspect the JSON-LD-style metadata representation
metadata_json = metadata.to_json()
print('Top-level metadata keys:')
print(sorted(metadata_json.keys()))

## 2. Data Overview
Review available dataset components, with special attention to record sets and their `@id` values.

Because Croissant entities should be referenced by `@id`, we extract those IDs explicitly and use variables for downstream processing.

In [ ]:
# Extract record set metadata from the JSON representation
record_sets_meta = metadata_json.get('recordSet', [])

print(f'Number of record sets declared in metadata: {len(record_sets_meta)}')

for i, rs in enumerate(record_sets_meta, start=1):
    rs_id = rs.get('@id', 'N/A')
    rs_name = rs.get('name', 'N/A')
    rs_description = rs.get('description', 'N/A')
    print(f'[{i}] record set @id: {rs_id}')
    print(f'    name: {rs_name}')
    print(f'    description: {rs_description}')
    
    fields = rs.get('field', [])
    print(f'    field count: {len(fields)}')
    for field in fields:
        field_id = field.get('@id', 'N/A')
        field_name = field.get('name', 'N/A')
        field_type = field.get('dataType', 'N/A')
        print(f'      - field @id: {field_id} | name: {field_name} | dataType: {field_type}')
    print()

In [ ]:
# Build a list of record set @id values for dynamic use later
record_set_ids = [rs.get('@id') for rs in record_sets_meta if rs.get('@id')]
print('Record set @id values found:')
print(record_set_ids)

The next cell shows how to inspect records from a specific record set by `@id`.

This cell is written defensively so the notebook remains runnable even when the dataset has no declared record sets.

In [ ]:
if record_set_ids:
    sample_record_set_id = record_set_ids[0]
    print(f'Showing up to 5 records from record set: {sample_record_set_id}')
    for idx, record in enumerate(dataset.records(record_set=sample_record_set_id)):
        print(record)
        if idx >= 4:
            break
else:
    print('No record sets are defined in this dataset metadata, so there are no tabular records to iterate over.')

## 3. Data Extraction
Load records from each available record set into pandas DataFrames.

All record sets are referenced by their `@id` values. The code uses variables throughout so it can be reused for other Croissant datasets with minimal changes.

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'Record set {record_set_id}: shape = {df.shape}')

if dataframes:
    example_record_set_id = next(iter(dataframes))
    print('\nExample record set @id:', example_record_set_id)
    print('Columns:')
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print('No DataFrames were created because no record sets were available.')

When no record sets are present, it is still useful to convert selected dataset-level metadata into a compact DataFrame for quick review.

In [ ]:
metadata_summary = {
    'dataset_name': dataset_name,
    'dataset_id': dataset_id,
    'version': metadata_json.get('version'),
    'license': metadata_json.get('license'),
    'date_published': metadata_json.get('datePublished'),
    'identifier': metadata_json.get('identifier'),
    'number_of_record_sets': len(record_sets_meta),
    'number_of_keywords': len(metadata_json.get('keywords', [])),
    'number_of_authors': len(metadata_json.get('author', [])),
    'number_of_citations': len(metadata_json.get('citation', [])),
    'number_of_visualizations': len(metadata_json.get('visualization', [])),
}

metadata_df = pd.DataFrame([metadata_summary])
display(metadata_df)

## 4. Exploratory Data Analysis (EDA)
This section applies simple exploratory analysis.

If record-set data exist, the notebook will:
- identify numeric columns,
- filter rows using a threshold,
- normalize a numeric field,
- optionally group by a categorical field.

If no record sets exist, the notebook performs EDA on the metadata itself, such as keyword, citation, and author counts.

In [ ]:
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    
    numeric_columns = df.select_dtypes(include='number').columns.tolist()
    print('Numeric columns found:', numeric_columns)
    
    if numeric_columns:
        numeric_field = numeric_columns[0]
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        
        print(f'Using numeric field: {numeric_field}')
        print(f'Filtering records where {numeric_field} > {threshold}')
        display(filtered_df.head())
        
        std_value = filtered_df[numeric_field].std()
        if pd.notna(std_value) and std_value != 0:
            normalized_col = f'{numeric_field}_normalized'
            filtered_df[normalized_col] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
            )
            print(f'Normalized column created: {normalized_col}')
            display(filtered_df[[numeric_field, normalized_col]].head())
        else:
            print('Standard deviation is zero or undefined; skipping normalization.')
        
        categorical_candidates = [
            c for c in df.columns
            if df[c].dtype == 'object' and df[c].nunique(dropna=True) > 0 and df[c].nunique(dropna=True) < len(df)
        ]
        print('Categorical grouping candidates:', categorical_candidates)
        
        if categorical_candidates:
            group_field = categorical_candidates[0]
            grouped_df = filtered_df.groupby(group_field, dropna=False)[numeric_field].mean(numeric_only=False).reset_index()
            print(f'Grouped mean of {numeric_field} by {group_field}:')
            display(grouped_df.head())
        else:
            print('No suitable categorical grouping column found.')
    else:
        print('No numeric columns found in the first available record set.')
else:
    print('No tabular record sets available. Running metadata-level EDA instead.')
    
    keywords = metadata_json.get('keywords', [])
    data_use_cases = metadata_json.get('dataUseCases', [])
    data_biases = metadata_json.get('dataBiases', [])
    
    metadata_eda_df = pd.DataFrame([
        {'metric': 'keyword_count', 'value': len(keywords)},
        {'metric': 'author_count', 'value': len(metadata_json.get('author', []))},
        {'metric': 'citation_count', 'value': len(metadata_json.get('citation', []))},
        {'metric': 'record_set_count', 'value': len(record_sets_meta)},
        {'metric': 'visualization_count', 'value': len(metadata_json.get('visualization', []))},
        {'metric': 'data_use_case_count', 'value': len(data_use_cases)},
        {'metric': 'data_bias_count', 'value': len(data_biases)},
    ])
    
    display(metadata_eda_df)
    
    threshold = 1
    filtered_df = metadata_eda_df[metadata_eda_df['value'] > threshold].copy()
    print(f'Metrics with value > {threshold}:')
    display(filtered_df)
    
    std_value = filtered_df['value'].std()
    if pd.notna(std_value) and std_value != 0:
        numeric_field = 'value'
        normalized_col = f'{numeric_field}_normalized'
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
        print('Normalized metadata metrics:')
        display(filtered_df[['metric', 'value', normalized_col]])
    else:
        print('Not enough variation to normalize metadata metrics.')

## 5. Visualization
Visualize either record-set data or metadata summaries, depending on what is available in the dataset.

In [ ]:
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    numeric_columns = df.select_dtypes(include='number').columns.tolist()
    
    if numeric_columns:
        numeric_field = numeric_columns[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field} in record set {record_set_id}')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()
    else:
        print('No numeric fields available for histogram visualization.')
else:
    viz_df = pd.DataFrame([
        {'metric': 'keywords', 'value': len(metadata_json.get('keywords', []))},
        {'metric': 'authors', 'value': len(metadata_json.get('author', []))},
        {'metric': 'citations', 'value': len(metadata_json.get('citation', []))},
        {'metric': 'record_sets', 'value': len(record_sets_meta)},
        {'metric': 'visualizations', 'value': len(metadata_json.get('visualization', []))},
    ])
    
    plt.figure(figsize=(8, 5))
    sns.barplot(data=viz_df, x='metric', y='value', palette='viridis')
    plt.title('Dataset Metadata Summary')
    plt.xlabel('Metric')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# Optional text-focused visualization: show keywords as a table for easy inspection
keywords = metadata_json.get('keywords', [])

if keywords:
    keywords_df = pd.DataFrame({'keyword': keywords})
    display(keywords_df)
else:
    print('No keywords found in metadata.')

## 6. Conclusion
This notebook loaded the Croissant dataset using `mlcroissant` and inspected its metadata in a reusable way.

### Key observations
- The dataset metadata is accessible and includes descriptive information such as name, description, authors, citations, keywords, and license.
- All access patterns in the notebook are based on Croissant entity `@id` values where applicable.
- In the provided metadata snapshot, the dataset declares **no record sets**, so there are currently no record-level tables to load into pandas from `dataset.records(...)`.
- Even without record sets, metadata-level exploration remains useful for understanding the dataset's scope, provenance, and intended use.

### Reuse idea
If this dataset is updated later to include one or more record sets, the same notebook can immediately load them because the extraction and analysis steps already use dynamically discovered record set `@id` values.